#  HousePrice 実習ノートブック
## 精度向上の工夫を考えよう（スライド p.94〜）

このノートブックは講習会スライドの **p.94 以降**に対応した実習用コードです。

---

### 本ノートブックの構成

各テーマは **3 パートで構成**されています。

| パート | マーク | 役割 |
|:---|:---:|:---|
| **DEMO** | 🔵 | スライドの解説・デモコード（**触らなくてOK**） |
| **QUIZ** | 🟡 | 穴埋め演習（**`___` の部分を埋めてください**） |
| **FREE** | 🟢 | 自由実装（**自分のアイデアを試す場所**） |

---

### セクション一覧

| # | セクション | スライド | テーマ |
|:---|:---|:---:|:---|
| 0 | セットアップ | — | ライブラリ・データ・ベースライン |
| 1 | EDA：基本統計量 | p.100〜101 | 平均・中央値・標準偏差 |
| 2 | EDA：データを可視化する | p.103〜105 | ヒストグラム・散布図 |
| 3 | EDA：欠損値を吟味する | p.106〜108 | 欠損の種類と補完戦略 |
| 4 | 前処理発展：カテゴリ別欠損補完 | p.109〜 | groupby を使った高度な補完 |
| 4 | 前処理：スケールと対数変換 | p.102〜103 | 標準化・正規化・対数変換 |
| 5 | 特徴量エンジニアリング | p.113〜117 | 組み合わせ・ドメイン知識 |
| 6 | 特徴量選択 | p.118〜123 | 多重共線性・相関フィルタ |
| 7 | まとめパイプライン＋評価 | — | 全工程を一本化して比較 |
| 8 | 提出ファイル作成 | — | Kaggle 提出用 CSV |


# 手法解説

---
## 0. セットアップ

> 🔵 **DEMO** ─ このセクションは触らずに上から順番に実行してください。

ライブラリを読み込み、データをロードして、**ベースライン RMSE** を確認します。  
この値が「改善前の基準スコア」になります。


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler

# グラフを日本語対応させる（Colab 環境）
!pip install japanize-matplotlib -q
import japanize_matplotlib

print('✅ ライブラリ読み込み完了')


In [ ]:
# ── データ読み込み ───────────────────────────────────────────────────────────
PATH = 'https://raw.githubusercontent.com/Toyoda05/kaggle-house-price-practice/main/'
train = pd.read_csv(PATH + 'train.csv')
test = pd.read_csv(PATH + 'test.csv')

print(f'train: {train.shape}  /  test: {test.shape}')
train.head()

In [ ]:
# ── ベースライン（前回と同じコード）─────────────────────────────────────────
test_ids = test['Id']

y = np.log1p(train['SalePrice'])
X = train.drop(columns=['SalePrice', 'Id'])
X_test = test.drop(columns=['Id']).copy()

all_data = pd.concat([X, X_test], axis=0).reset_index(drop=True)
numeric_cols     = all_data.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = all_data.select_dtypes(include=['object']).columns

for col in numeric_cols:
    all_data[col] = all_data[col].fillna(all_data[col].median())
for col in categorical_cols:
    all_data[col] = all_data[col].fillna('Missing')

all_data = pd.get_dummies(all_data, drop_first=True)

X_processed      = all_data.iloc[:len(X)].copy()
X_test_processed = all_data.iloc[len(X):].copy()

X_train, X_valid, y_train, y_valid = train_test_split(
    X_processed, y, test_size=0.2, random_state=42
)

model_baseline = LinearRegression()
model_baseline.fit(X_train, y_train)
valid_pred_baseline = model_baseline.predict(X_valid)

rmse_baseline = np.sqrt(mean_squared_error(y_valid, valid_pred_baseline))
print(f'\n🔵 【ベースライン RMSE】 {rmse_baseline:.5f}  ← この値を超えることを目指そう！')

---
## 1. EDA：基本統計量を確認しよう（スライド p.100〜101）

> **EDA（Exploratory Data Analysis）** ＝ データを可視化・集計して特徴を探る作業

まずは **基本統計量（平均・中央値・最頻値・分散・標準偏差）** を見て、データ全体の概要を把握します。


### 🔵 DEMO ─ 基本統計量を見てみよう


In [ ]:
# describe() で一覧表示
train.describe().round(1)

In [ ]:
# 平均・中央値・最頻値を並べて比較し、ヒストグラムに重ねて表示する
target = 'SalePrice'

mean_val   = train[target].mean()
median_val = train[target].median()
mode_val   = train[target].mode()[0]

print(f'平均値  (mean)  : ${mean_val:,.0f}')
print(f'中央値  (median): ${median_val:,.0f}')
print(f'最頻値  (mode)  : ${mode_val:,.0f}')

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(train[target], bins=60, color='steelblue', alpha=0.7, label='SalePrice')
ax.axvline(mean_val,   color='red',    linestyle='--', label=f'Mean   {mean_val:,.0f}')
ax.axvline(median_val, color='orange', linestyle='--', label=f'Median {median_val:,.0f}')
ax.axvline(mode_val,   color='green',  linestyle='--', label=f'Mode   {mode_val:,.0f}')
ax.set_xlabel('SalePrice ($)')
ax.set_title('SalePrice の分布（平均・中央値・最頻値）')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 標準偏差ランキング（ばらつきが大きい特徴量を確認）
cols_numeric = train.select_dtypes(include='number').drop(columns=['Id', 'SalePrice'])
std_df = cols_numeric.std().sort_values(ascending=False).head(10).reset_index()
std_df.columns = ['特徴量', '標準偏差']
print('標準偏差が大きい上位 10 特徴量')
print(std_df.to_string(index=False))

### 🟡 QUIZ ─ 穴埋め演習：別の特徴量で統計量を確認しよう

`GrLivArea`（地上居住面積）について、平均・中央値・標準偏差を計算し、  
ヒストグラムを描いてみましょう。`___` を埋めてコードを完成させてください。


In [ ]:
# 🟡 QUIZ：GrLivArea の基本統計量を確認する
col = 'GrLivArea'

# ── ❶ 統計量を計算する ──────────────────────────────────────
mean_gl   = train[col].mean()           # 平均値
median_gl = train[col].median()         # 中央値
std_gl    = train[col].std()            # 標準偏差

print(f'平均値   : {mean_gl:.1f}')
print(f'中央値   : {median_gl:.1f}')
print(f'標準偏差 : {std_gl:.1f}')

# ── ❷ ヒストグラムを描く ────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(train[col], bins=50, color='steelblue', alpha=0.7)   # bins の数を決めよう（例: 50）
ax.axvline(mean_gl,   color='red',    linestyle='--', label='Mean')   # 平均線
ax.axvline(median_gl, color='orange', linestyle='--', label='Median') # 中央線
ax.set_xlabel(col)
ax.set_title(f'{col} の分布')
ax.legend()
plt.tight_layout()
plt.show()


### 🟢 FREE ─ 自由実装：自分が気になる特徴量を調べよう

気になる特徴量を自分で選んで、基本統計量とヒストグラムを確認してみましょう。


In [ ]:
# 🟢 FREE：好きな特徴量を選んで統計量を確認しよう
# ── 使えそうな特徴量の例 ──────────────────────────────────────────────────
# 'LotArea' (土地面積), 'YearBuilt' (建築年), 'OverallQual' (総合品質スコア)
# 'TotalBsmtSF' (地下面積), 'GarageArea' (ガレージ面積) など

# ↓↓↓ ここに自由にコードを書こう ↓↓↓

my_col = 'LotArea'   # ← 変えてみよう

print(train[my_col].describe())

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(train[my_col].dropna(), bins=50, color='mediumpurple', alpha=0.7)
ax.set_xlabel(my_col)
ax.set_title(f'{my_col} の分布')
plt.tight_layout()
plt.show()
print('データの最大・最小は、どのような意味を持つのだろうか？')
print('標準偏差はどのように活用できるのだろうか？')

---
## 2. EDA：データを可視化してみよう（スライド p.103〜105）

グラフを描いて、**目的変数（SalePrice）と各特徴量の関係**を視覚的に探りましょう。


### 🔵 DEMO ─ データのスケールを考慮しよう！


In [ ]:
# SalePrice の分布と対数変換（スライド p.103）
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].hist(train['SalePrice'], bins=50, color='steelblue')
axes[0, 0].set_title('SalePrice（変換前）')
axes[0, 0].set_xlabel('SalePrice ($)')

axes[0, 1].hist(np.log1p(train['SalePrice']), bins=50, color='green')
axes[0, 1].set_title('log1p(SalePrice)（変換後）')
axes[0, 1].set_xlabel('log1p(SalePrice)')

stats.probplot(train['SalePrice'], plot=axes[1, 0])
axes[1, 0].set_title('Q-Q plot（変換前）')

stats.probplot(np.log1p(train['SalePrice']), plot=axes[1, 1])
axes[1, 1].set_title('Q-Q plot（変換後）')

plt.tight_layout()
plt.show()

print(f'変換前の歪度: {train["SalePrice"].skew():.3f}')
print(f'変換後の歪度: {np.log1p(train["SalePrice"]).skew():.3f}')
print('歪度（わいど）が 0 に近いほど正規分布に近い形です')

### 🔵 DEMO ─ SalePrice の分布・散布図・箱ひげ図


In [ ]:
# SalePrice との相関が高い上位12特徴量を散布図で確認（スライド p.104〜105）
y_log = np.log1p(train['SalePrice'])
X_num = train.select_dtypes(include=['int64', 'float64']).drop(columns=['SalePrice', 'Id'])

corr_with_y = X_num.corrwith(y_log).abs().sort_values(ascending=False)
top12 = corr_with_y.head(12).index

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(top12):
    axes[i].scatter(train[col], y_log, alpha=0.3, s=10, color='steelblue')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('log(SalePrice)')
    axes[i].set_title(f'{col}\n(r={corr_with_y[col]:.2f})')

plt.tight_layout()
plt.show()

In [ ]:
# カテゴリ変数との関係（OverallQual × SalePrice の箱ひげ図）
fig, ax = plt.subplots(figsize=(10, 5))
groups = [train[train['OverallQual'] == q]['SalePrice']
          for q in sorted(train['OverallQual'].unique())]
ax.boxplot(groups, labels=sorted(train['OverallQual'].unique()))
ax.set_xlabel('OverallQual（総合品質スコア）')
ax.set_ylabel('SalePrice ($)')
ax.set_title('OverallQual ごとの SalePrice 分布')
plt.tight_layout()
plt.show()
print('品質スコアが上がると価格はどう変わりますか？ばらつきは？')

### 🟡 QUIZ ─ 穴埋め演習：指定特徴量の散布図を描こう

`YearBuilt`（建築年）と `log1p(SalePrice)` の散布図を描き、  
相関係数 r を計算して、タイトルに表示してみましょう。


In [ ]:
# 🟡 QUIZ：YearBuilt vs log1p(SalePrice) の散布図
col   = 'YearBuilt'
y_log = np.log1p(train['SalePrice'])

# ── ❶ 相関係数を計算する ─────────────────────────────────────
r = train[col].corr(y_log)   # 相関係数を計算するには何を引数にすればよい？

# ── ❷ 散布図を描く ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(train[col], y_log, alpha=0.4, s=15, color='steelblue')  # y 軸は何？
ax.set_xlabel(col)
ax.set_ylabel('log1p(SalePrice)')
ax.set_title(f'{col} vs log1p(SalePrice)  (r = {r:.3f})')
plt.tight_layout()
plt.show()

### 🟢 FREE ─ 自由実装：好きな特徴量で散布図・箱ひげ図を描こう


In [ ]:
# 🟢 FREE：自由に可視化してみよう
# ── 試してみると面白い特徴量 ───────────────────────────────────────────────
# 散布図向き : 'GrLivArea', 'TotalBsmtSF', 'GarageArea', 'LotArea'
# 箱ひげ図向き: 'Neighborhood', 'BldgType', 'KitchenQual'（カテゴリ変数）

# ↓↓↓ ここに自由にコードを書こう ↓↓↓

my_col = 'GrLivArea'   # ← 変えてみよう
y_log  = np.log1p(train['SalePrice'])

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(train[my_col], y_log, alpha=0.4, s=15, color='darkorange')
ax.set_xlabel(my_col)
ax.set_ylabel('log1p(SalePrice)')
r_my = train[my_col].corr(y_log)
ax.set_title(f'{my_col} vs log1p(SalePrice)  (r = {r_my:.3f})')
plt.tight_layout()
plt.show()
print('Sales Price との関係性は？')
print('外れ値はどうだろうか？')

---
## 3. 前処理：欠損値を吟味しよう（スライド p.106〜108）

欠損値には大きく **2 種類**あります。

| 種類 | 意味 | 対処法の例 |
|:---|:---|:---|
| **データが本当に存在しない** | 計測漏れ・入力ミス | 中央値・平均値・最頻値で補完 |
| **「該当設備なし」を示す NA** | PoolQC=NA → プールなし | 新しいカテゴリ「None」として扱う |


### 🔵 DEMO ─ 欠損の可視化と「NA = 設備なし」の確認


In [ ]:
# 欠損割合を棒グラフで可視化する
def plot_missing(df, title='train'):
    missing = df.isnull().sum()
    missing = missing[missing > 0].sort_values(ascending=False)
    missing_pct = (missing / len(df) * 100).round(1)

    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.barh(missing.index, missing_pct.values, color='salmon')
    ax.set_xlabel('欠損率 (%)')
    ax.set_title(f'欠損値の割合（{title}）')
    for bar, val in zip(bars, missing_pct.values):
        ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
                f'{val}%', va='center', fontsize=9)
    plt.tight_layout()
    plt.show()

plot_missing(train, 'train')

In [ ]:
# data_description.txt より：NA が「設備なし」を意味する列の例（1部）
na_means_none = [
    'PoolQC', 'MiscFeature', 'Alley', 'Fence',
    'FireplaceQu', 'GarageType', 'GarageFinish'
]

print('=== NA が「設備なし」を意味する列 ===')
for col in na_means_none:
    if col in train.columns:
        n_miss = train[col].isnull().sum()
        pct    = n_miss / len(train) * 100
        print(f'  {col:<20}: 欠損数={n_miss:4d} ({pct:.1f}%)')

### 🟡 QUIZ ─ 穴埋め演習：補完方法を比較しよう

`LotFrontage`（前面道路幅）は「真の欠損」です。  
平均値・中央値・最頻値で補完したときの **分散** を比べて、最もデータの性質を保つ方法はどれか確認しましょう。


In [ ]:
# 🟡 QUIZ：LotFrontage の欠損補完方法を比較する
col      = 'LotFrontage'
original = train[col].copy()

# ── ❶ 3種類の補完を行う ──────────────────────────────────────
fill_mean   = original.fillna(original.mean())              # 平均値で補完
fill_median = original.fillna(original.median())            # 中央値で補完
fill_mode   = original.fillna(original.mode()[0])           # 最頻値で補完

# ── ❷ 分散を比較して結果を表示する ─────────────────────────────
print(f'{col} の分析')
print(f'  欠損数  : {original.isnull().sum()}')
print(f'  平均値  : {original.mean():.1f}  →  補完後の分散: {fill_mean.var():.1f}')
print(f'  中央値  : {original.median():.1f}  →  補完後の分散: {fill_median.var():.1f}')
print(f'  最頻値  : {original.mode()[0]:.1f}  →  補完後の分散: {fill_mode.var():.1f}')

# ── ❸ ヒストグラムで補完前後を比較する ─────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(original.dropna(), bins=30, color='salmon',    label='元データ（欠損除外）')
axes[0].set_title('補完前（欠損除く）')
axes[1].hist(fill_median, bins=30, color='steelblue', label='中央値補完')  # 中央値補完の結果を可視化
axes[1].set_title('中央値補完後')
for ax in axes:
    ax.set_xlabel(col)
plt.tight_layout()
plt.show()

### 🟢 FREE ─ 自由実装：別の列の欠損を調べてみよう


In [ ]:
# 🟢 FREE：気になる欠損列を選んで補完を試してみよう
# ── 欠損が多い列の例 ────────────────────────────────────────────────────
# 数値: 'GarageYrBlt', 'MasVnrArea'  → 中央値補完が基本
# 文字: 'FireplaceQu', 'GarageFinish' → NA = 「設備なし」なので 'None' 補完

# ↓↓↓ ここに自由にコードを書こう ↓↓↓

my_col = 'GarageYrBlt'  # ← 変えてみよう
print(f'{my_col} の欠損数: {train[my_col].isnull().sum()}')
print(train[my_col].describe())

# 補完後の確認
filled = train[my_col].fillna(train[my_col].median())
print(f'補完後の欠損数: {filled.isnull().sum()}')

## 4. 前処理発展：カテゴリ別の代表値で欠損補完しよう

### 🔵DEMO ─ 土地別の中央値で LotFrontage を補完

In [ ]:
# ── カテゴリ別代表値補完のデモ ──────────────────────────────────────────
col      = 'LotFrontage'   # 前面道路幅（欠損 147 件）
group_by = 'Neighborhood'  # 地域ごとにグループ化

# ❶ グループ別中央値を確認する
group_median = train.groupby(group_by)[col].median().sort_values()
print('=== Neighborhood 別 LotFrontage 中央値（一部） ===')
print(group_median.head(5).to_string())
print('...')
print(group_median.tail(5).to_string())

fig, ax = plt.subplots(figsize=(12, 5))
group_median.plot(kind='bar', ax=ax, color='steelblue', alpha=0.8)
ax.set_xlabel('Neighborhood')
ax.set_ylabel('LotFrontage 中央値 (ft)')
ax.set_title('Neighborhood 別 LotFrontage の中央値（地域差が大きい！）')
ax.axhline(train[col].median(), color='red', linestyle='--', label=f'全体中央値 {train[col].median():.0f}ft')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ❷ transform('median') で各行に「そのグループの中央値」を対応させて補完
lot_before = train[col].copy()

fill_global = lot_before.fillna(lot_before.median())                        # 全体中央値補完
fill_group  = lot_before.fillna(                                             # カテゴリ別補完
    train.groupby(group_by)[col].transform('median')
)

print(f'補完前  の欠損数 : {lot_before.isnull().sum()}')
print(f'全体中央値補完後 : {fill_global.isnull().sum()}')
print(f'グループ別補完後 : {fill_group.isnull().sum()}')

# ❸ 補完方法ごとの分布を比較する
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

axes[0].hist(lot_before.dropna(), bins=30, color='salmon',    alpha=0.8)
axes[0].set_title(f'補完前（欠損 {lot_before.isnull().sum()} 件除外）')

axes[1].hist(fill_global, bins=30, color='steelblue', alpha=0.8)
axes[1].set_title('全体中央値で補完')

axes[2].hist(fill_group,  bins=30, color='seagreen',  alpha=0.8)
axes[2].set_title('Neighborhood 別中央値で補完')

for ax in axes:
    ax.set_xlabel(col)
plt.suptitle('LotFrontage の補完方法比較', y=1.02)
plt.tight_layout()
plt.show()

print(f'\n分散の比較')
print(f'  全体中央値補完 : {fill_global.var():.2f}')
print(f'  グループ別補完 : {fill_group.var():.2f}')
print('→ グループ別補完のほうが分散が大きい（元のデータのばらつきを保てている）')

### 🟡QUIZ ─ 穴埋め演習：GarageYrBlt をカテゴリ別に補完しよう

`GarageYrBlt`（ガレージ建設年）は `GarageType`（ガレージ種別）と関係が深いと考えられます。

`GarageType` 別の中央値を使って欠損を補完し、全体中央値補完と結果を比べましょう。

In [ ]:
# 🟡 QUIZ：GarageYrBlt の欠損をカテゴリ別中央値で補完する
col      = 'GarageYrBlt'
group_by = 'GarageType'

yr_original = train[col].copy()
print(f'欠損数: {yr_original.isnull().sum()} 件')

# ── ❶ GarageType 別の中央値を確認する ──────────────────────────────
group_med = train.groupby(group_by)[col].median()   # ← メソッド名を入れよう
print('\nGarageType 別 GarageYrBlt 中央値:')
print(group_med)

# ── ❷ 全体中央値補完とカテゴリ別補完を実装する ───────────────────────
fill_global = yr_original.fillna(yr_original.median())   # ← 全体の中央値を入れよう

fill_group  = yr_original.fillna(
    train.groupby(group_by)[col].transform('median')     # ← 集計関数名（文字列）を入れよう
)

print(f'\n全体中央値補完後の欠損: {fill_global.isnull().sum()}')
print(f'グループ別補完後の欠損 : {fill_group.isnull().sum()}')

# ── ❸ 補完後の分散を比較する ─────────────────────────────────────────
print(f'\n分散の比較')
print(f'  全体中央値補完 : {fill_global.var():.2f}')
print(f'  グループ別補完 : {fill_group.var():.2f}')

# ── ❹ ヒストグラムで可視化する ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(fill_global, bins=30, color='steelblue', alpha=0.8)  # ← 全体中央値補完の結果
axes[0].set_title('全体中央値で補完')
axes[1].hist(fill_group,  bins=30, color='seagreen',  alpha=0.8)  # ← グループ別補完の結果
axes[1].set_title('GarageType 別中央値で補完')
for ax in axes:
    ax.set_xlabel(col)
plt.suptitle('GarageYrBlt の補完方法比較', y=1.02)
plt.tight_layout()
plt.show()


### 🟢 FREE ─ 自由実装：自分でグループ列と補完列を選ぼう

どのカテゴリ列をグループにすると、どの数値列をうまく補完できるか、自分で仮説を立てて試してみましょう。

In [ ]:
# 🟢 FREE：カテゴリ別補完を自分でデザインしよう
# ── アイデア例 ─────────────────────────────────────────────────────────────
# ● col='MasVnrArea',  group_by='MasVnrType'  （外壁材の種類 → 外壁面積）
# ● col='LotFrontage', group_by='LotConfig'   （土地区画の形状 → 道路幅）
# ● col='LotFrontage', group_by='MSZoning'    （用途地域 → 道路幅）

# ↓↓↓ ここに自由にコードを書こう ↓↓↓

my_col      = 'MasVnrArea'   # ← 補完したい数値列
my_group_by = 'MasVnrType'   # ← グループ化するカテゴリ列

print(f'欠損数: {train[my_col].isnull().sum()} 件')
print(f'\n{my_group_by} 別 {my_col} 中央値:')
print(train.groupby(my_group_by)[my_col].median())

fill_my = train[my_col].fillna(
    train.groupby(my_group_by)[my_col].transform('median')
)
print(f'\n補完後の欠損数: {fill_my.isnull().sum()}')

# 分散を比較してみよう
fill_global_my = train[my_col].fillna(train[my_col].median())
print(f'全体中央値補完の分散 : {fill_global_my.var():.2f}')
print(f'グループ別補完の分散  : {fill_my.var():.2f}')

# グループ間で中央値に差があると、カテゴリ別補完が特に有効！
# → 差が小さければ、全体中央値補完でも十分かもしれない

---
## 5. 特徴量エンジニアリング（スライド p.113〜117）

### 考え方
1. **既存特徴量を組み合わせる** → 比・積・差で新たな意味を持つ変数を作る（スライド p.115）
2. **ドメイン知識を使う** → 不動産の知識から「高級物件シグナル」などを作る（スライド p.116）

> 💡 **作り方の指針（スライド p.117）**
> - ターゲットと強い相関が期待できるドメイン知識は何か？
> - 既存特徴量の比・積・差で意味ある量が作れないか？
> - 特定の条件でグループ化したときの統計量が使えないか？


### 🔵 DEMO ─ 特徴量の組み合わせ・ドメイン知識による特徴量作成


In [ ]:
# ① 特徴量を組み合わせる（スライド p.115）
train_fe = train.copy()

# 例1：「建物の実質的な新しさ」= 建築年 + リフォーム年
train_fe['Age_Plus_Remod'] = train_fe['YearBuilt'] + train_fe['YearRemodAdd']

# 例2：「土地の資産価値」= 土地面積 × 総合品質スコア
train_fe['LotArea_x_Qual'] = train_fe['LotArea'] * train_fe['OverallQual']

# 例3：「総床面積」= 1階 ＋ 2階
train_fe['TotalSF_manual'] = train_fe['1stFlrSF'] + train_fe['2ndFlrSF']

new_cols = ['Age_Plus_Remod', 'LotArea_x_Qual', 'TotalSF_manual']

# 新特徴量と SalePrice の相関を確認
corr_new = train_fe[new_cols + ['SalePrice']].corr()['SalePrice'].drop('SalePrice')
print('新特徴量と SalePrice の相関:')
print(corr_new.round(3))

orig_corr = train[['YearBuilt', 'YearRemodAdd', 'LotArea', 'OverallQual',
                    '1stFlrSF', '2ndFlrSF', 'SalePrice']].corr()['SalePrice'].drop('SalePrice')
print('\n元の特徴量と SalePrice の相関（参考）:')
print(orig_corr.round(3))

In [ ]:
# ② ドメイン知識を使う（スライド p.116）
# 「プールまたは暖炉がある = 高級住宅シグナル」
train_fe['Luxury_Signal'] = (
    (train_fe['PoolArea'] > 0) | (train_fe['Fireplaces'] > 0)
).astype(int)

fig, ax = plt.subplots(figsize=(7, 4))
group0 = train_fe[train_fe['Luxury_Signal'] == 0]['SalePrice']
group1 = train_fe[train_fe['Luxury_Signal'] == 1]['SalePrice']
ax.boxplot([group0, group1],
           labels=['Luxury=0\n（プールも暖炉もなし）', 'Luxury=1\n（プールor暖炉あり）'])
ax.set_ylabel('SalePrice ($)')
ax.set_title('Luxury_Signal ごとの SalePrice')
plt.tight_layout()
plt.show()

print(f'Luxury=0 の中央価格: ${group0.median():,.0f}')
print(f'Luxury=1 の中央価格: ${group1.median():,.0f}')

### 🟡 QUIZ ─ 穴埋め演習：新しい特徴量を作って相関を確認しよう

「築年数（HouseAge）」は住宅価格と関係が強そうです。  
`YrSold - YearBuilt` で計算し、SalePrice との相関係数と散布図を確認しましょう。


In [ ]:
# 🟡 QUIZ：築年数 HouseAge を作成して効果を確認する
train_fe2 = train.copy()

# ── ❶ 築年数を計算する ──────────────────────────────────────
train_fe2['HouseAge'] = train_fe2['YrSold'] - train_fe2['YearBuilt']   # 売却年 - 建築年

# ── ❷ SalePrice との相関係数を計算する ─────────────────────────
r = train_fe2['HouseAge'].corr(train_fe2['SalePrice'])
print(f'HouseAge と SalePrice の相関係数: {r:.3f}')

# ── ❸ 散布図を描く ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(train_fe2['HouseAge'], np.log1p(train_fe2['SalePrice']),
           alpha=0.4, s=15, color='steelblue')
ax.set_xlabel('HouseAge（築年数）')
ax.set_ylabel('log1p(SalePrice)')
ax.set_title(f'HouseAge vs log1p(SalePrice)  (r = {r:.3f})')
plt.tight_layout()
plt.show()

# ── ❹ 別の組み合わせも作ってみよう ───────────────────────────
# リフォームからの経過年数 = YrSold - YearRemodAdd
train_fe2['RemodAge'] = train_fe2['YrSold'] - train_fe2['YearRemodAdd']
r2 = train_fe2['RemodAge'].corr(train_fe2['SalePrice'])
print(f'RemodAge と SalePrice の相関係数: {r2:.3f}')

# ── ❺ 考察 ───────────────────────────────────────────────────
# Q: HouseAge と RemodAge、どちらが SalePrice とより強く相関していますか？


### 🟢 FREE ─ 自由実装：自分だけの特徴量を作ってみよう

**スライド p.117 の指針**を参考に、オリジナルの特徴量を考えてみましょう。

```
ヒント例：
  TotalBath     = FullBath + 0.5*HalfBath + BsmtFullBath + 0.5*BsmtHalfBath
  LotRatio      = GrLivArea / LotArea   （敷地に対する居住面積の割合）
  QualxArea     = OverallQual * GrLivArea
  HasGarage     = (GarageCars > 0).astype(int)
```


In [ ]:
# 🟢 FREE：オリジナル特徴量を作ってみよう
train_fe3 = train.copy()

# ↓↓↓ ここに自由にコードを書こう ↓↓↓

# 例：バスルーム合計
train_fe3['TotalBath'] = (train_fe3['FullBath']
                         + 0.5 * train_fe3['HalfBath']
                         + train_fe3['BsmtFullBath']
                         + 0.5 * train_fe3['BsmtHalfBath'])

r = train_fe3['TotalBath'].corr(train_fe3['SalePrice'])
print(f'TotalBath と SalePrice の相関: {r:.3f}')

# さらに別の特徴量を追加してみよう！

---
## 6. 特徴量選択
特徴量が増えすぎると、**本当に重要な特徴量が埋もれて**しまいます。  
2 つの観点で不要な特徴量を削除しましょう。

1. **多重共線性**：特徴量同士の相関が強い → 一方を削除
2. **目的変数との相関が弱い**：ノイズになるリスクがある → 閾値で機械的に削除


### 🔵 DEMO ─ 多重共線性のヒートマップ・相関フィルタの可視化


In [ ]:
# ① 多重共線性の確認
corr = train[['GarageCars','GarageArea','TotalBsmtSF',
           '1stFlrSF','GrLivArea','TotRmsAbvGrd']].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("代表的な特徴量間の相関ヒートマップ\n（強い相関 = 多重共線性の疑い）")

print(f'GarageCars と GarageArea の相関係数: {train["GarageCars"].corr(train["GarageArea"]):.3f}')

In [ ]:
# ② 目的変数との相関が弱い特徴量を可視化
y_log = np.log1p(train['SalePrice'])
X_num = train.select_dtypes(include='number').drop(columns=['SalePrice', 'Id'])
corr_with_target = X_num.corrwith(y_log).sort_values()

colors = ['#e74c3c' if abs(v) < 0.1 else '#2ecc71' for v in corr_with_target]

fig, ax = plt.subplots(figsize=(10, 12))
corr_with_target.plot(kind='barh', ax=ax, color=colors)
ax.axvline(x= 0.1, color='black', linestyle='--', linewidth=1, label='threshold +0.1')
ax.axvline(x=-0.1, color='black', linestyle='--', linewidth=1, label='threshold -0.1')
ax.set_title('SalePrice との相関係数\n（赤 = |r| < 0.1、削除候補）', fontsize=14)
ax.set_xlabel('Pearson 相関係数')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 削除候補リストを確認する
THRESHOLD = 0.1
weak_features = corr_with_target[corr_with_target.abs() < THRESHOLD].index.tolist()

print(f'|r| < {THRESHOLD} の特徴量（削除候補）: {len(weak_features)} 個')
print(weak_features)

### 🟡 QUIZ ─ 穴埋め演習：多重共線性ペアを自分で探してみよう

全数値特徴量の相関行列から、**相関係数が 0.8 以上のペア**を抽出してみましょう。


In [ ]:
# 🟡 QUIZ：相関係数 0.8 以上の特徴量ペアを抽出する
X_num = train.select_dtypes(include='number').drop(columns=['SalePrice', 'Id'])

# ── ❶ 相関行列を計算する ─────────────────────────────────────
corr_all = X_num.corr()   # .corr() で相関行列を求める

# ── ❷ 上三角行列だけ取り出して高相関ペアを探す ──────────────────
high_corr_pairs = []
cols = corr_all.columns

for i in range(len(cols)):
    for j in range(i + 1, len(cols)):
        r_val = corr_all.iloc[i, j]
        if abs(r_val) >= 0.8:   # 閾値（0.8）を入れよう
            high_corr_pairs.append((cols[i], cols[j], round(r_val, 3)))

print(f'相関係数 |r| >= 0.8 のペア: {len(high_corr_pairs)} 組')
print()
for col_a, col_b, r_val in sorted(high_corr_pairs, key=lambda x: -abs(x[2])):
    print(f'  {col_a:<20} ×  {col_b:<20}  r = {r_val}')

# ── ❸ 考察 ───────────────────────────────────────────────────
# Q: 最も相関が強いペアはどれですか？どちらを残しますか？


### 🟢 FREE ─ 自由実装：閾値を変えて削除する列数を調整してみよう


In [ ]:
# 🟢 FREE：THRESHOLD を変えて、削除候補の特徴量数がどう変わるか確認しよう

# ↓↓↓ ここに自由にコードを書こう ↓↓↓

y_log  = np.log1p(train['SalePrice'])
X_num2 = train.select_dtypes(include='number').drop(columns=['SalePrice', 'Id'])
corr2  = X_num2.corrwith(y_log).abs()

# 調べたい閾値を並べよう
for thresh in [0.05, 0.10, 0.15, 0.20]:
    n_drop = (corr2 < thresh).sum()
    print(f'閾値 {thresh:.2f} → 削除候補: {n_drop:2d} 特徴量')

# さらに: 削除したほうがいい特徴量・残したほうがいい特徴量を
# ドメイン知識から考えてみよう